In [0]:
from delta.tables import DeltaTable
import os
import sys

# Ensure Python can find utils.py from the parent directory
project_dir = "/Workspace/Users/ranayuvrajsingh1111@gmail.com/Olist ecomerce ETL"
if project_dir not in sys.path:
  sys.path.append(project_dir)

import utils
from pyspark.sql import functions as F

# ==========================================
# 1. EXTRACT GOLD FACT SALES
# ==========================================
print("Extracting gold_fact_sales for aggregation...")
df_fact = spark.read.table("ecommerce_lakehouse.gold.gold_fact_sales")

# ==========================================
# 2. AGGREGATE METRICS BY DATE, CATEGORY, & STATE
# ==========================================
df_summary = df_fact.groupBy("purchase_date", "category_name", "seller_state").agg(
    F.countDistinct("order_id").alias("total_orders"),
    F.count("order_item_id").alias("total_items_sold"),
    F.sum("price").alias("total_product_revenue"),
    F.sum("freight_value").alias("total_freight_revenue"),
    F.sum("total_item_amount").alias("total_gross_revenue"),
    F.avg("price").alias("avg_item_price"),
    F.sum(F.when(F.col("is_delayed") == True, 1).otherwise(0)).alias(
        "delayed_orders_count"
    ),
)

# Sort chronologically for clean storage and querying
df_transformed = df_summary.orderBy(
    F.col("purchase_date").desc(), F.col("total_gross_revenue").desc()
)

# ==========================================
# 3. WRITE TO GOLD LAYER
# ==========================================
target_table_name = "ecommerce_lakehouse.gold.gold_daily_sales_summary"
print(f"Writing data to Gold table: {target_table_name}")

df_transformed.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(target_table_name)

print("Successfully generated gold_daily_sales_summary!")

Extracting gold_fact_sales for aggregation...
Writing data to Gold table: ecommerce_lakehouse.gold.gold_daily_sales_summary
Successfully generated gold_daily_sales_summary!
